# Wstęp do Sztucznej Inteligencji

Przed rozpoczęciem pracy z notatnikiem zmień jego nazwę zgodnie z wzorem: `NrAlbumu_Nazwisko_Imie_PoprzedniaNazwa`.

Przed wysłaniem notatnika upewnij się, że rozwiązałeś wszystkie zadania/ćwiczenia.

# Temat: Wnioskowanie oparte na wiedzy niepewnej. Wnioskowanie bayesowskie
Zapoznaj się z treścią niniejszego notatnika czytając i wykonując go komórka po komórce. Wykonaj napotkane zadania/ćwiczenia.

In [1]:
import pymc as pm
import arviz as az
import numpy as np

print('Pymc version:', pm.__version__)
print('Arviz version:', az.__version__)
print('Numpy version:', np.__version__)

Pymc version: 5.28.2
Arviz version: 0.23.4
Numpy version: 2.4.3


## Zadanie 1  (obowiązkowe, 5pkt.)
Zapożyczone z ćwiczeń do wykładu [Arabas, Cichosz](http://wazniak.mimuw.edu.pl/index.php?title=Sztuczna_inteligencja/SI_%C4%86wiczenia_4)

### Zamodeluj i odpowiedz na pytania.

W śledztwie dotyczącym zabójstwa inspektor Bayes rozważa dwie hipotezy:

- $H_1$ główny podejrzany zabił,
- $H_2$ główny podejrzany nie zabił, 

oraz następujące możliwe fakty:

- $E_1$ na miejscu zbrodni znaleziono odciski palców głównego podejrzanego,
- $E_2$ główny podejrzany nie ma alibi na czas popełnienia zabójstwa,
- $E_3$ główny podejrzany miał motyw zabicia ofiary,
- $E_4$ główny podejrzany był widziany w sądziedztwie miejsca, w którym mieszka nielegalny handlarz bronią,
- $E_5$ świadek zbrodni podał rysopis zabójcy nie pasujący do głównego podejrzanego. 

Zależności między takimi faktami a hipotezami opisują następujące prawdopodobieństwa:

$P(E_1|H_1)=0.7,\qquad P(E_1|H_2)=0.3,$

$P(E_2|H_1)=0.8,\qquad P(E_2|H_2)=0.4,$

$P(E_3|H_1)=0.9,\qquad P(E_3|H_2)=0.5,$

$P(E_4|H_1)=0.4,\qquad P(E_4|H_2)=0.2,$

$P(E_5|H_1)=0.2,\qquad P(E_5|H_2)=0.4.$ 

__W którym przypadku prawdopodobieństwo popełnienia zabójstwa byłoby największe?__

1. Gdyby znaleziono na miejscu zbrodni jego odciski palców.
2. Gdyby stwierdzono, że nie miał alibi i miał motyw.
3. Gdyby znaleziono na miejscu zbrodni jego odciski palców oraz stwierdzono, że był widziany w sąsiedztwie miejsca, w którym mieszka nielegalny handlarz bronią, ale świadek zbrodni podał rysopis zabójcy nie pasujący do głównego podejrzanego.

### TWÓJ PROGRAM:

In [2]:
import pymc as pm

with pm.Model() as model:
	did_kill = pm.Bernoulli("did_kill", p=0.5)

	fingerprints = pm.Deterministic(
		"fingerprints",
		pm.math.switch(did_kill, 0.7, 0.3),
	)

	no_alibi = pm.Deterministic(
		"no_alibi",
		pm.math.switch(did_kill, 0.8, 0.4),
	)

	motive = pm.Deterministic(
		"motive",
		pm.math.switch(did_kill, 0.9, 0.5),
	)
	seen_at_scene = pm.Deterministic(
		"seen_at_scene",
		pm.math.switch(did_kill, 0.4, 0.2),
	)

	inconsistent_description = pm.Deterministic(
		"inconsistent_description",
		pm.math.switch(did_kill, 0.2, 0.4),
	)

	# Mam dobry komputer to czemu nie
	trace = pm.sample(20_000, chains=64, cores=64, return_inferencedata=True)


def posterior_values(name: str):
	return trace.posterior[name].stack(sample=("chain", "draw")).values


did_kill_samples = posterior_values("did_kill")
fingerprints_samples = posterior_values("fingerprints")
no_alibi_samples = posterior_values("no_alibi")
motive_samples = posterior_values("motive")
seen_at_scene_samples = posterior_values("seen_at_scene")
inconsistent_description_samples = posterior_values("inconsistent_description")


def conditional_probability(hypothesis_samples, evidence_samples):
	return (hypothesis_samples * evidence_samples).sum() / evidence_samples.sum()


p_did_kill_given_fingerprints = conditional_probability(
	did_kill_samples,
	fingerprints_samples,
)

p_did_kill_given_no_alibi_and_motive = conditional_probability(
	did_kill_samples,
	no_alibi_samples * motive_samples,
)

p_did_kill_given_fingerprints_seen_and_inconsistent_description = conditional_probability(
	did_kill_samples,
	fingerprints_samples * seen_at_scene_samples * inconsistent_description_samples,
)

print(f"1. P(did_kill | fingerprints): {p_did_kill_given_fingerprints:.16f}")
print(f"2. P(did_kill | no_alibi, motive): {p_did_kill_given_no_alibi_and_motive:.16f}")
print(
	"3. P(did_kill | fingerprints, seen_at_scene, inconsistent_description): "
	f"{p_did_kill_given_fingerprints_seen_and_inconsistent_description:.16f}"
)

Multiprocess sampling (4 chains in 4 jobs)
BinaryGibbsMetropolis: [did_kill, fingerprints]


Output()

Sampling 4 chains for 1_000 tune and 20_000 draw iterations (4_000 + 80_000 draws total) took 4 seconds.


1. P(did_kill | fingerprints): 0.6996940515598354
2. P(did_kill | no_alibi, motive): 0.7823023016055993
3. P(did_kill | fingerprints, seen_at_scene, inconsistent_description): 0.6996940515598354


### ODPOWIEDŹ:

1. Odciski palców -> prawdopodobieństwo, że podejrzany zabił wynosi **0.7000839932805376**
2. Brak alibi + motyw -> prawdopodobieństwo, że podejrzany zabił wynosi **0.7826767408901600** (największe)
3. Odciski + widziany + niepasujący rysopis -> prawdopodobieństwo, że podejrzany zabił wynosi **0.7000839932805373**

## Zadanie 2  (obowiązkowe, 5pkt.)

### Zamodeluj i odpowiedz na pytania.
System alarmowy w mieszkaniu, reaguje na włamania oraz, niestety, również na drobne trzęsienia (ziemi). Sąsiedzi John i Mary są umówieni, żeby zadzwonić do właściciela gdy usłyszą alarm. John jest nadgorliwy i bierze różne zdarzenia (np. dzwonek telefonu) za sygnał alarmowy (i wtedy zawsze dzwoni). Mary rozpoznaje alarm poprawnie, lecz często słucha głośnej muzyki i może go w ogóle nie usłyszeć. 

Sieć przekonań dla systemu alarmowego wygląda następująco:
![bsiec.PNG](../images/bsiec.PNG)

__Jakie jest prawdopodobieństwo, że:__
1. włączy się alarm?
2. doszło do włamanie jeśli wiadom, że włączył się alarm?
3. zdarzyło się trzęsienie ziemi jeśli wiadomo, żę włączył się alarm?
1. w razie włamania ktoś zadzwoni?
2. zawiadomienie o włamaniu jest fałszywe?
3. rozległ się alarm, przy czym nie wystąpiło ani trzęsienie ziemi ani włamanie, ale oboje John i Mary zadzwonili? (prawd. bezwarunkowe)

TWÓJ PROGRAM:

In [9]:
import pymc as pm

with pm.Model() as model:
	burglary = pm.Bernoulli("burglary", p=0.01)
	earthquake = pm.Bernoulli("earthquake", p=0.02)

	alarm_probability = pm.math.switch(
		burglary,
		pm.math.switch(earthquake, 0.95, 0.94),
		pm.math.switch(earthquake, 0.29, 0.001),
	)

	alarm = pm.Bernoulli("alarm", p=alarm_probability)

	john_calls = pm.Bernoulli("john_calls", p=pm.math.switch(alarm, 0.9, 0.05))
	mary_calls = pm.Bernoulli("mary_calls", p=pm.math.switch(alarm, 0.7, 0.01))

	# Mam dobry komputer to czemu nie
	trace = pm.sample(20_000, chains=64, cores=64, return_inferencedata=True)


def posterior_values(name: str):
	return trace.posterior[name].stack(sample=("chain", "draw")).values


burglary_samples = posterior_values("burglary")
earthquake_samples = posterior_values("earthquake")
alarm_samples = posterior_values("alarm")
john_calls_samples = posterior_values("john_calls")
mary_calls_samples = posterior_values("mary_calls")

someone_called = (john_calls_samples > 0) | (mary_calls_samples > 0)

p_alarm = alarm_samples.mean()

p_burglary_given_alarm = (burglary_samples * alarm_samples).sum() / alarm_samples.sum()
p_earthquake_given_alarm = (earthquake_samples * alarm_samples).sum() / alarm_samples.sum()

p_someone_calls_given_burglary = (someone_called * burglary_samples).sum() / burglary_samples.sum()

p_false_alarm_call = ((1 - burglary_samples) * someone_called).sum() / someone_called.sum()

p_alarm_no_burglary_no_earthquake_john_and_mary_call = (
	(
			(alarm_samples == 1)
			& (burglary_samples == 0)
			& (earthquake_samples == 0)
			& (john_calls_samples == 1)
			& (mary_calls_samples == 1)
	).mean()
)

print(f"1. Alarm probability: {p_alarm:.16f}")
print(f"2. P(burglary | alarm): {p_burglary_given_alarm:.16f}")
print(f"3. P(earthquake | alarm): {p_earthquake_given_alarm:.16f}")
print(f"4. P(someone calls | burglary): {p_someone_calls_given_burglary:.16f}")
print(f"5. P(false alarm call): {p_false_alarm_call:.16f}")
print(
	"6. P(alarm, no burglary, no earthquake, John calls, Mary calls): "
	f"{p_alarm_no_burglary_no_earthquake_john_and_mary_call:.16f}"
)

Multiprocess sampling (64 chains in 64 jobs)
BinaryGibbsMetropolis: [burglary, earthquake, alarm, john_calls, mary_calls]


Output()

Sampling 64 chains for 1_000 tune and 20_000 draw iterations (64_000 + 1_280_000 draws total) took 137 seconds.


1. Alarm probability: 0.0167000000000000
2. P(burglary | alarm): 0.5719966317365269
3. P(earthquake | alarm): 0.3813622754491018
4. P(someone calls | burglary): 0.9166987154834243
5. P(false alarm call): 0.8751780477586929
6. P(alarm, no burglary, no earthquake, John calls, Mary calls): 0.0006132812500000


### ODPOWIEDŹ:

1. Włączy się alarm -> prawdopodobieństwo wynosi **0.0167000000000000** ≈ **1.67%**.
2. Doszło do włamania, jeśli włączył się alarm -> **0.5719966317365269** ≈ **57.19%**.
3. Zdarzyło się trzęsienie ziemi, jeśli włączył się alarm -> **0.3813622754491018** ≈ **38.13%**.
4. W razie włamania ktoś zadzwoni -> **0.9166987154834243** ≈ **91.66%**.
5. Zawiadomienie o włamaniu jest fałszywe -> **0.8751780477586929** ≈ **87.51%**.
6. Alarm się rozległ, nie było ani włamania ani trzęsienia ziemi, a John i Mary zadzwonili -> **0.0006132812500000** ≈ **0.0613%**.

__UWAGA:__ Zwróć uwagę na wielkości podanych prawdopodobieńst aby dobarć odpowiednią liczbę symulacji.

&copy; Cracow University of Technology